# TabPFN (Tabular Prior-data Fitted Network)

Notebook 2 established the LightGBM ensemble as the best model. This notebook explores whether **TabPFN** — a tabular foundation model that uses in-context learning rather than gradient-boosted trees — can help, either as a standalone predictor or as a more diverse additional ensemble member.

**TabPFN** (Tabular Prior-data Fitted Network) is a transformer pre-trained once on millions of synthetic tabular tasks, which then performs prediction in a single forward pass via in-context learning. It "reads" the training rows as context and predicts the test rows directly, with no per-task gradient training. Its inductive bias comes from the synthetic prior it was trained on, not from fitting our DEL data (fundamentally different approach from the trees), which could contribute complementary signal to an ensemble. This notebook tests that hypothesis.

>### **Setup and constraints**
>
> <u>Capacity limits:</u>  
> TabPFN-3 trades rows against features: up to 1,000,000 × 200, 100,000 × 2,000, or 1,000 × 20,000. I use the **100,000 × 2,000 mode** — a 100k subsample at 2,000 features — which keeps near-complete fingerprint information while running efficiently on available hardware.
>
> <u>Fingerprint representation — Set 1:</u>  
> Unlike the tree ensembles (which average a model *per fingerprint* across all seven), TabPFN takes a single concatenated feature matrix. I use **Set 1 (ECFP6 + RDK + ATOMPAIR)**, the decorrelation-optimal trio from Notebook 2: the strongest individual fingerprint (ECFP6) plus the two most-decorrelated strong fingerprints (RDK, ATOMPAIR). This gives TabPFN the same rich, complementary representation that worked best for the trees, rather than a single fingerprint.  
>
> **Why not all seven (set1+2+3)?** Concatenating seven 2,048-dim fingerprints (14,336 dims) before PCA, then compressing to TabPFN's 2,000-feature budget, would force far heavier compression and blend in the weaker, more redundant fingerprints (e.g. the Morgan duplicates ECFP4/ECFP6, and the weaker Set 3 trio). Set 1 keeps the three strongest decorrelated fingerprints with lighter compression; cleaner, more information-dense input for the capacity-limited model.
>
> <u>Feature compression:</u>  
> The concatenated Set 1 fingerprints are compressed to **2,000 principal components** via PCA (fit on training, applied identically to test), retaining ~95% of the variance (as seen below).
>
> <u>Practicalities:</u>  
> TabPFN-3 runs far faster on GPU; this was run on Google Colab (**L4 GPU**).

---

## 1. Data and feature preparation

Load the full training and test sets, and build the **Set 1** feature matrix (ECFP6 + RDK + ATOMPAIR concatenated), compressed to **2,000 principal components** via PCA (fit on the training set, applied identically to the test set), retaining ~95% of the variance.

For TabPFN's 100,000 × 2,000 capacity mode, a **100,000-compound training subsample** is taken (all hits plus a sample of non-hits). The test set (all 339,258 compounds) is then predicted in full.

### <u>Setup</u>

In [ ]:
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA

def parse_fp(s): return np.array(s.split(","), dtype=np.float32)
def fp_matrix(df, fp): return csr_matrix(np.stack(df[fp].map(parse_fp).to_numpy()))

train = pd.read_parquet("../data/raw/crosstalk_train.parquet")
test  = pd.read_parquet("../data/raw/crosstalk_test_inputs.parquet")
y_train = train["DELLabel"].to_numpy().astype(int)
print("loaded:", train.shape, test.shape)

loaded: (375595, 16) (339258, 12)


### <u>Build set 1 (ECFP6, RDK, ATOMPAIR) with upsampled hit sampling:</u>

In [71]:
SET1 = ["ECFP6", "RDK", "ATOMPAIR"]

# upsampled 100k subsample: ALL hits + non-hits to fill to 100k (~29% hit rate)
rng = np.random.default_rng(42)
hit_idx = np.where(y_train==1)[0]; non_idx = np.where(y_train==0)[0]
n_hit = len(hit_idx)                      # all 28,778 hits
n_non = 100_000 - n_hit                   # 71,222 non-hits
sub = np.concatenate([hit_idx, rng.choice(non_idx, n_non, replace=False)])
rng.shuffle(sub)
ys = y_train[sub]
np.save("y_set1_upsampled.npy", ys)
print(f"subsample: {len(sub)} rows, {ys.mean():.1%} hits")

# build set1 PCA features
Xtr = np.hstack([fp_matrix(train, fp).toarray() for fp in SET1])   # 375595 x 6144
Xte = np.hstack([fp_matrix(test, fp).toarray() for fp in SET1])
pca = PCA(n_components=2000, random_state=42).fit(Xtr)
print(f"set1: {pca.explained_variance_ratio_.sum():.1%} variance retained")
np.save("train_set1_upsampled.npy", pca.transform(Xtr)[sub].astype(np.float32))
np.save("test_set1_pca2000.npy", pca.transform(Xte).astype(np.float32))
print("saved set1 + labels")

subsample: 100000 rows, 28.8% hits
set1: 95.0% variance retained
saved set1 + labels


---

## 2. Upload the files to Google Drive for TabPFN on Google Colab
Upload these 4 files:
- train_set1_upsampled.npy
- test_set1_pca2000.npy
- y_set1_upsampled.npy
- crosstalk_test_inputs.parquet

---
## 3. In Google Colab's L4 GPU, run below:

```python
#install + GPU:
!pip install tabpfn pyarrow scikit-learn -q
import torch
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


#mount Drive + token:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = "/content/drive/MyDrive/CodingJourney/CROSSTALK/TabPFN/"
with open(BASE + "token.txt") as f: #saved my token in a text file
    os.environ["TABPFN_TOKEN"] = f.read().strip()
print("token loaded")


#train set 1 + predict + save results:
import numpy as np, pandas as pd
from tabpfn import TabPFNClassifier
from tqdm.auto import tqdm

ys = np.load(BASE + "y_set1_upsampled.npy")
xs = np.load(BASE + "train_set1_upsampled.npy")
Xte = np.load(BASE + "test_set1_pca2000.npy")

print(f"training set1 on {xs.shape}, {ys.mean():.1%} hits")
clf = TabPFNClassifier(device="cuda", ignore_pretraining_limits=True)
clf.fit(xs, ys)
tabpfn_set1 = np.concatenate([clf.predict_proba(Xte[i:i+5000])[:,1]
                              for i in tqdm(range(0, len(Xte), 5000), desc="set1")])
np.save(BASE + "new_tabpfn_set1.npy", tabpfn_set1)
print("DONE:", tabpfn_set1.shape)
```

---

## 4. Building TabPFN submissions
Back in VSCode, upload the resulting `tabpfn_set1.npy` from Google Drive

Then using the same diversity re-ranking scheme as Notebook 2, I build the plain TabPFN set1 submission plus mild/moderate diversity variants (β=0.2, 0.4) as penalty-robust hedges.


In [25]:
import numpy as np, pandas as pd
from rdkit import DataStructs
from rdkit.DataStructs import ExplicitBitVect

p1 = np.load("new_tabpfn_set1.npy")

# ---- diversity re-ranking setup (same MMR scheme as Notebook 2) ----
def parse_fp(s): return np.array(s.split(","), dtype=np.float32)

# ECFP4 similarity matrix for diversity (field-standard for Tanimoto diversity)
test = pd.read_parquet("../input_data/raw/crosstalk_test_inputs.parquet")
test_ecfp4 = np.stack(test["ECFP4"].map(parse_fp).to_numpy())
print("test_ecfp4:", test_ecfp4.shape)

def to_bitvect(count_row):
    bv = ExplicitBitVect(len(count_row))
    for i in np.nonzero(count_row)[0]:
        bv.SetBit(int(i))
    return bv

def diverse_rerank(scores, count_matrix, top_pool=2000, beta=0.5):
    """Greedy MMR re-ranking: balance model score against similarity to already-selected."""
    pool = np.argsort(scores)[::-1][:top_pool]
    bvs = [to_bitvect(count_matrix[i]) for i in pool]
    selected, sel_bvs, remaining = [], [], list(range(len(pool)))
    while remaining:
        best_j, best_val = None, -1e9
        for j in remaining:
            sim = max(DataStructs.BulkTanimotoSimilarity(bvs[j], sel_bvs)) if sel_bvs else 0.0
            val = scores[pool[j]] - beta * sim
            if val > best_val:
                best_val, best_j = val, j
        selected.append(pool[best_j])
        sel_bvs.append(bvs[best_j])
        remaining.remove(best_j)
    return selected

# ---- build TabPFN submissions (plain + diversity variants) ----
ids = pd.read_parquet("../input_data/raw/crosstalk_test_inputs.parquet", columns=["RandomID"])["RandomID"].values

def apply_beta(scores, sim, beta):
    order = diverse_rerank(scores, sim, top_pool=2000, beta=beta)
    s = np.zeros(len(scores))
    for r, idx in enumerate(order): s[idx] = len(order) - r
    return s

# plain TabPFN set1 submission
pd.DataFrame({"RandomID": ids, "DELLabel": p1}).to_csv("submission_tabpfn_set1.csv", index=False)
print("saved submission_tabpfn_set1.csv")

# diversity-reranked variants
for beta in [0.2, 0.4]:
    sc = apply_beta(p1, test_ecfp4, beta)
    pd.DataFrame({"RandomID": ids, "DELLabel": sc}).to_csv(f"submission_tabpfn_set1_beta{beta}.csv", index=False)
    print(f"saved submission_tabpfn_set1_beta{beta}.csv")

test_ecfp4: (339258, 2048)
saved submission_tabpfn_set1.csv
saved submission_tabpfn_set1_beta0.2.csv
saved submission_tabpfn_set1_beta0.4.csv


### <u>Leaderboard results in Kaggle</u>
After submission of the TabPFN CSVs, the public score results were:

| Submission | Hits@200 |
|---|---|
| TabPFN | 1 |
| TabPFN (β=0.2) | 1 |
| TabPFN (β=0.4) | 1 |

**TabPFN underperformed as a standalone predictor** — 1 hit in the top-200, far below the LightGBM ensemble's 9

- The diversity variants (β=0.2, 0.4) made no difference (all scored 1)
- With only ~1 hit in the top-200 initially, the underlying ranking is too weak for diversity re-ranking to meaningfully reshape (reshuffling noise, not surfacing hits)
- Could be due to single-fingerprint constraint — TabPFN sees one concatenated Set 1 matrix versus the trees' seven per-fingerprint models
- Though the gap is too large to attribute to representation alone

---

## 5. Conclusion

This notebook tested whether **TabPFN**, a tabular foundation model with a fundamentally different inductive bias from the trees, could improve on the LightGBM pipeline.

**As a standalone predictor, TabPFN underperformed**
- It scored 1 hit on the leaderboard, versus the tuned LightGBM ensemble's 9.

**But the value of a different architecture is not its solo score — it is whether it is *complementary***
- A model can rank poorly overall yet still rank *some* true hits well that the trees miss
- If it does so on *different* compounds, fusing it with the strong model could surface hits the strong model alone misses

---
---

## Next Steps

TabPFN is weak on its own, but its predictions come from a genuinely different inductive bias than the trees so they may be *decorrelated* from LightGBM. 

Decorrelation is exactly what makes ensemble fusion work: a weak-but-different model could help in a combination if it ranks some hits well where the strong model does not.

**Notebook 4** tests this directly. It measures how correlated TabPFN actually is with the tree models, then combines them through _rank fusion_ (max-rank, mean-rank, and weighted blends), _stacking_ (a learned meta-model), and _multi-model fusion_ (three or more models together). The question throughout: can TabPFN's complementary signal, or any combination of models, push past the single LightGBM's performance?